In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML

ACCENT = '#4f46e5'
GREEN  = '#059669'
PINK   = '#e11d48'
ORANGE = '#d97706'
BLUE   = '#0284c7'
TEXT   = '#1e293b'
AC = {'Caminata': BLUE, 'Carrera': PINK, 'Reposo': GREEN, 'Ciclismo': ORANGE}
PT = dict(layout=dict(
    paper_bgcolor='#ffffff',
    plot_bgcolor='#f8fafc',
    font=dict(color=TEXT, family='Inter,sans-serif'),
    xaxis=dict(gridcolor='#e2e8f0', zerolinecolor='#e2e8f0'),
    yaxis=dict(gridcolor='#e2e8f0', zerolinecolor='#e2e8f0'),
    margin=dict(l=40, r=20, t=50, b=40),
    legend=dict(bgcolor='#ffffff', bordercolor='#e2e8f0', borderwidth=1)
))

In [ ]:
np.random.seed(42)
N = 1000
acts = np.random.choice(['Caminata','Carrera','Reposo','Ciclismo'], N, p=[0.30,0.25,0.25,0.20])
HRP = {'Caminata':(95,15),    'Carrera':(160,20), 'Reposo':(65,10),   'Ciclismo':(130,18)}
STP = {'Caminata':(5500,1200),'Carrera':(9000,1500),'Reposo':(300,150),'Ciclismo':(4000,800)}
CAP = {'Caminata':(280,60),   'Carrera':(600,80),  'Reposo':(60,15),   'Ciclismo':(450,70)}
SPP = {'Caminata':(97.5,1.0), 'Carrera':(96.0,1.5),'Reposo':(98.5,0.5),'Ciclismo':(97.0,1.2)}
hr = [max(45, min(220, np.random.normal(*HRP[a]))) for a in acts]
st = [max(0,  int(np.random.normal(*STP[a])))      for a in acts]
ca = [max(20, np.random.normal(*CAP[a]))            for a in acts]
sp = [min(100,max(88, np.random.normal(*SPP[a])))   for a in acts]
df = pd.DataFrame({
    'Actividad':         acts,
    'Frec_Cardiaca':     np.round(hr, 1),
    'Conteo_Pasos':      st,
    'Calorias_Quemadas': np.round(ca, 1),
    'SpO2':              np.round(sp, 1),
    'Edad':              np.random.randint(18, 65, N),
    'Nivel_Atletico':    np.random.choice(['Principiante','Intermedio','Avanzado'], N, p=[0.4,0.35,0.25])
})
print(f'Conjunto de datos listo: {df.shape[0]} registros x {df.shape[1]} variables')

In [ ]:
display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
.dbh {
  background: linear-gradient(135deg, #eef2ff 0%, #f0fdf4 100%);
  border: 2px solid #c7d2fe; border-radius: 16px;
  padding: 28px 36px; margin-bottom: 20px;
  position: relative; overflow: hidden;
}
.dbh::before {
  content: ''; position: absolute; top: 0; left: 0; right: 0; height: 4px;
  background: linear-gradient(90deg, #4f46e5, #059669, #e11d48);
}
.dbt { font-family: Inter,sans-serif; font-size: 26px; font-weight: 700; color: #1e1b4b; margin: 0 0 6px; }
.dbs { font-family: Inter,sans-serif; font-size: 13px; color: #475569; }
.badge { display: inline-block; background: #e0e7ff; color: #3730a3;
  border: 1px solid #a5b4fc; border-radius: 20px;
  padding: 3px 12px; font-size: 11px; font-weight: 600; margin: 8px 4px 0 0; }
.kpi-grid { display: grid; grid-template-columns: repeat(4,1fr); gap: 14px; margin: 20px 0; }
.kpi { background: #ffffff; border: 1px solid #e2e8f0;
  border-radius: 12px; padding: 18px 20px; box-shadow: 0 1px 3px rgba(0,0,0,.06); }
.kl { font-family: Inter,sans-serif; font-size: 11px; color: #64748b;
  text-transform: uppercase; letter-spacing: .08em; margin-bottom: 6px; }
.kv { font-family: Inter,sans-serif; font-size: 26px; font-weight: 700; color: #1e293b; }
.kd { font-family: Inter,sans-serif; font-size: 11px; margin-top: 3px; color: #94a3b8; }
.st { font-family: Inter,sans-serif; font-size: 15px; font-weight: 600; color: #1e293b;
  margin: 20px 0 10px; padding-left: 10px; border-left: 3px solid #4f46e5; }
.ctrl-box { background: #f8fafc; border: 1px solid #e2e8f0;
  border-radius: 8px; padding: 16px 20px; margin-bottom: 16px; }
.ctrl-title { font-family: Inter,sans-serif; font-size: 12px; font-weight: 600;
  color: #64748b; text-transform: uppercase; letter-spacing: .08em; margin-bottom: 10px; }
</style>
<div class='dbh'>
  <div class='dbt'>Panel de Salud Deportiva con Wearables</div>
  <div class='dbs'>Analisis interactivo de metricas fisiologicas &middot; Ciencia de Datos II &middot; 2026</div>
  <span class='badge'>Analisis Exploratorio</span>
  <span class='badge'>Regresion Lineal</span>
  <span class='badge'>Deteccion de Anomalias</span>
  <span class='badge'>Segmentacion por Actividad</span>
</div>
"""))

In [ ]:
display(HTML("<div class='ctrl-box'><div class='ctrl-title'>Controles del Panel</div></div>"))

sl = {'description_width': '140px'}
af  = widgets.SelectMultiple(
        options=['Caminata','Carrera','Reposo','Ciclismo'],
        value=['Caminata','Carrera','Reposo','Ciclismo'],
        description='Actividades:',
        style=sl, layout=widgets.Layout(width='300px'))
hrs = widgets.IntRangeSlider(
        value=[40,220], min=40, max=220, step=5,
        description='Frec. Cardiaca:',
        style=sl, layout=widgets.Layout(width='400px'))
alf = widgets.RadioButtons(
        options=['Todos','Principiante','Intermedio','Avanzado'],
        value='Todos', description='Nivel atletico:', style=sl)
ct  = widgets.ToggleButtons(
        options=['Distribucion','Correlacion','Regresion','Diagrama de Caja'],
        description='Vista principal:',
        style={'description_width':'120px','button_width':'130px'})

display(widgets.HBox([af, widgets.VBox([hrs, alf])]))
display(ct)

In [ ]:
ko = widgets.Output()
mo = widgets.Output()
bo = widgets.Output()
display(ko)
display(mo)
display(bo)


def gf(at, hr, al):
    m = (df.Actividad.isin(list(at)) &
         (df.Frec_Cardiaca >= hr[0]) & (df.Frec_Cardiaca <= hr[1]))
    if al != 'Todos':
        m &= (df.Nivel_Atletico == al)
    return df[m].copy()


def rk(f):
    n   = len(f)
    ah  = f.Frec_Cardiaca.mean()
    as_ = f.Conteo_Pasos.mean()
    cr  = f[['Conteo_Pasos','Frec_Cardiaca']].corr().iloc[0,1]
    pn  = n / len(df) * 100
    cc  = '#059669' if abs(cr) > 0.5 else '#d97706'
    cl  = 'Alta' if abs(cr) > 0.5 else 'Baja'
    return (
        f"<div class='kpi-grid'>"
        f"<div class='kpi' style='border-top:3px solid {ACCENT}'>"
        f"<div class='kl'>Registros activos</div><div class='kv'>{n:,}</div>"
        f"<div class='kd'>{pn:.1f}% del total</div></div>"
        f"<div class='kpi' style='border-top:3px solid {PINK}'>"
        f"<div class='kl'>Frec. Cardiaca Media</div>"
        f"<div class='kv' style='color:{PINK}'>{ah:.1f} bpm</div></div>"
        f"<div class='kpi' style='border-top:3px solid {BLUE}'>"
        f"<div class='kl'>Pasos Promedio</div>"
        f"<div class='kv' style='color:{BLUE}'>{as_:,.0f}</div></div>"
        f"<div class='kpi' style='border-top:3px solid {GREEN}'>"
        f"<div class='kl'>Correlacion Pasos-FC</div>"
        f"<div class='kv' style='color:{GREEN}'>{cr:+.3f}</div>"
        f"<div class='kd' style='color:{cc}'>{cl} correlacion</div></div></div>"
    )


def md(f):
    fig = make_subplots(rows=1, cols=2,
        subplot_titles=['Distribucion Frecuencia Cardiaca', 'Distribucion Conteo de Pasos'])
    for a in f.Actividad.unique():
        s = f[f.Actividad == a]
        c = AC.get(a, '#333')
        fig.add_trace(go.Histogram(x=s.Frec_Cardiaca, name=a,
                                   marker_color=c, opacity=0.75), row=1, col=1)
        fig.add_trace(go.Histogram(x=s.Conteo_Pasos, name=a, showlegend=False,
                                   marker_color=c, opacity=0.75), row=1, col=2)
    fig.update_layout(**PT['layout'], height=400, barmode='overlay',
                      title_text='Distribucion de Variables por Actividad')
    return fig


def mc(f):
    fig = go.Figure()
    for a in f.Actividad.unique():
        s = f[f.Actividad == a]
        c = AC.get(a, '#333')
        fig.add_trace(go.Scatter(
            x=s.Conteo_Pasos, y=s.Frec_Cardiaca, mode='markers', name=a,
            marker=dict(color=c, size=6, opacity=0.65,
                        line=dict(color='white', width=0.5)),
            hovertemplate='<b>%{fullData.name}</b><br>Pasos: %{x:,}<br>FC: %{y:.1f} bpm<extra></extra>'
        ))
    fig.update_layout(**PT['layout'], height=400,
                      xaxis_title='Conteo de Pasos',
                      yaxis_title='Frecuencia Cardiaca (bpm)',
                      title='Correlacion: Pasos vs Frecuencia Cardiaca')
    return fig


def mr(f):
    fig = go.Figure()
    x = f.Conteo_Pasos.values.astype(float)
    y = f.Frec_Cardiaca.values.astype(float)
    for a in f.Actividad.unique():
        s = f[f.Actividad == a]
        c = AC.get(a, '#333')
        fig.add_trace(go.Scatter(
            x=s.Conteo_Pasos, y=s.Frec_Cardiaca,
            mode='markers', name=a,
            marker=dict(color=c, size=6, opacity=0.5)))
    if len(x) > 1:
        cs = np.polyfit(x, y, 1)
        xl = np.linspace(x.min(), x.max(), 200)
        yp = np.polyval(cs, x)
        r2 = 1 - np.sum((y - yp)**2) / np.sum((y - y.mean())**2)
        fig.add_trace(go.Scatter(
            x=xl, y=np.polyval(cs, xl), mode='lines',
            name=f'Recta de regresion (R2={r2:.3f})',
            line=dict(color='#dc2626', width=2.5)))
        fig.add_annotation(
            x=0.05, y=0.95, xref='paper', yref='paper',
            text=f'y = {cs[0]:.4f}x + {cs[1]:.2f}<br>R2 = {r2:.4f}',
            showarrow=False,
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor=ACCENT, borderwidth=1,
            font=dict(color=TEXT, size=12), align='left')
    fig.update_layout(**PT['layout'], height=400,
                      xaxis_title='Conteo de Pasos',
                      yaxis_title='Frecuencia Cardiaca (bpm)',
                      title='Regresion Lineal: Pasos -> Frecuencia Cardiaca')
    return fig


def mb(f):
    fig = make_subplots(rows=1, cols=2,
        subplot_titles=['Frecuencia Cardiaca por Actividad',
                        'Calorias Quemadas por Actividad'])
    for a in ['Caminata', 'Carrera', 'Reposo', 'Ciclismo']:
        s = f[f.Actividad == a]
        if len(s) == 0:
            continue
        c = AC.get(a, '#333')
        fig.add_trace(go.Box(y=s.Frec_Cardiaca, name=a,
                             marker_color=c, boxmean=True), row=1, col=1)
        fig.add_trace(go.Box(y=s.Calorias_Quemadas, name=a, showlegend=False,
                             marker_color=c, boxmean=True), row=1, col=2)
    fig.update_layout(**PT['layout'], height=400)
    return fig


def raf(f):
    cats = ['Frec_Cardiaca','Conteo_Pasos','Calorias_Quemadas','SpO2','Edad']
    lbls = ['Frec. Cardiaca','Pasos','Calorias','SpO2','Edad']
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type':'polar'}, {'type':'xy'}]],
        subplot_titles=[
            'Perfil Fisiologico por Actividad (normalizado)',
            'Deteccion de Valores Atipicos (IQR)'
        ])
    gm = f.groupby('Actividad')[cats].mean()
    nm = (gm - gm.min()) / (gm.max() - gm.min() + 1e-9)
    for a, r in nm.iterrows():
        c = AC.get(a, '#333')
        v = list(r.values) + [r.values[0]]
        l = lbls + [lbls[0]]
        fig.add_trace(go.Scatterpolar(
            r=v, theta=l, fill='toself', name=a,
            line_color=c, opacity=0.8), row=1, col=1)
    od = []
    for col in ['Frec_Cardiaca', 'Conteo_Pasos', 'Calorias_Quemadas']:
        q1, q3 = f[col].quantile([0.25, 0.75])
        iq = q3 - q1
        no = ((f[col] < q1-1.5*iq) | (f[col] > q3+1.5*iq)).sum()
        od.append({'Variable': col.replace('_',' '),
                   'Atipicos': no, 'Pct': no/len(f)*100})
    odf = pd.DataFrame(od)
    fig.add_trace(go.Bar(
        x=odf.Variable, y=odf.Atipicos,
        marker_color=[PINK, BLUE, ORANGE],
        text=[f'{p:.1f}%' for p in odf.Pct],
        textposition='outside', showlegend=False), row=1, col=2)
    fig.update_layout(**PT['layout'], height=380)
    fig.update_polars(
        bgcolor='#f8fafc',
        radialaxis=dict(visible=True, gridcolor='#e2e8f0'),
        angularaxis=dict(gridcolor='#e2e8f0'))
    return fig


CHS = {
    'Distribucion':     md,
    'Correlacion':      mc,
    'Regresion':        mr,
    'Diagrama de Caja': mb
}


def upd(at, hr, al, V):
    f = gf(at, hr, al)
    with ko:
        ko.clear_output(wait=True)
        if len(f) > 0:
            display(HTML(rk(f)))
        else:
            display(HTML('<p style="color:#dc2626;font-family:Inter">Sin datos con los filtros actuales.</p>'))
    with mo:
        mo.clear_output(wait=True)
        display(HTML(f'<div class="st">{V}</div>'))
        CHS[V](f).show(config={'displayModeBar': True, 'displaylogo': False})
    with bo:
        bo.clear_output(wait=True)
        display(HTML('<div class="st">Perfil Fisiologico y Deteccion de Anomalias</div>'))
        raf(f).show(config={'displayModeBar': False})


out = widgets.interactive_output(upd, {'at': af, 'hr': hrs, 'al': alf, 'V': ct})
display(out)

In [ ]:
display(HTML("""
<div style='background:#f0f9ff;border:1px solid #bae6fd;
     border-radius:12px;padding:16px 22px;margin-top:18px;
     font-family:Inter,sans-serif;font-size:12px;color:#475569'>
  <strong style='color:#1e40af'>Fundacion Universitaria Compensar</strong>
  &middot; Programacion para Ciencia de Datos II &middot; 2026-1
  &middot; Daylher Fabian Rodriguez Pena<br>
  <span style='color:#0369a1'>Conjunto de datos:</span>
  Monitoreo de Salud Deportiva con Wearables (Kaggle)
  &middot; <span style='color:#0369a1'>Herramientas:</span>
  Python &middot; NumPy &middot; Pandas &middot; Plotly &middot; ipywidgets &middot; Voila
</div>
"""))